In [2]:
import pandas as pd

df = pd.read_csv("Dataset_v1.0_raw.csv")
df.head()

,Year,Quarter,Month (Sort Order),MonthIndex,MonthSin,MonthCos,isHolidayMonth,Account ID Text,Region,Segment,...,RevenueLag2,RevenueLag3,RevenueLag12,Rolling3MRevenue,AvgDealSize,AvgDealSizeRolling3M,TopProductId,OpportunityConcentrationHHI,IsActiveLast3M,MonthlyTotalRevenue
0,2020,1,1,0,0.0,1.0,1,1,Central,Strategic,...,NaN,NaN,NaN,NaN,36354.7927,NaN,2,0.206126,0,181773.9636
1,2020,1,1,0,0.0,1.0,1,10,East,Strategic,...,NaN,NaN,NaN,NaN,31394.9485,NaN,11,0.104318,0,345344.4335
2,2020,1,1,0,0.0,1.0,1,100,Central,Strategic,...,NaN,NaN,NaN,38634.3833,36048.8728,38634.3833,4,0.078004,1,504684.2193
3,2020,1,1,0,0.0,1.0,1,101,West,Large,...,NaN,NaN,NaN,NaN,33166.7633,NaN,17,0.103648,0,364834.3964
4,2020,1,1,0,0.0,1.0,1,102,Central,Large,...,NaN,NaN,NaN,NaN,35696.1947,NaN,9,0.059517,0,678227.6987


In [3]:
!pip install catboost

In [4]:
#1. Cargar el dataset
import pandas as pd
import numpy as np
from catboost import CatBoostRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.preprocessing import LabelEncoder, StandardScaler

print("Librerías cargadas correctamente")

import pandas as pd

df = pd.read_csv("Dataset_v1.0_raw.csv")
df.shape

Librerías cargadas correctamente


(23919, 21)

In [5]:
# 2. Limpieza y preparación de datos

# Buena práctica: renombrar columnas
df = df.rename(columns={
    "Year": "year",
    "Quarter": "quarter",
    "Month (Sort Order)": "month",
    "MonthIndex": "month_index",
    "MonthSin": "month_sin",
    "MonthCos": "month_cos",
    "isHolidayMonth": "is_holiday_month",
    "Account ID Text": "account_id",
    "Region": "region",
    "Segment": "segment",
    "RevenueLag1": "revenue_lag_1",
    "RevenueLag2": "revenue_lag_2",
    "RevenueLag3": "revenue_lag_3",
    "RevenueLag12": "revenue_lag_12",
    "Rolling3MRevenue": "rolling_3m_revenue",
    "AvgDealSize": "avg_deal_size",
    "AvgDealSizeRolling3M": "avg_deal_size_rolling_3m",
    "TopProductId": "top_product_id",
    "OpportunityConcentrationHHI": "opportunity_concentration_hhi",
    "IsActiveLast3M": "is_active_last_3m",
    "MonthlyTotalRevenue": "monthly_total_revenue"
})

# Blindaje de tipos
df["account_id"] = df["account_id"].astype(str)

for col in ["region", "segment"]:
    df[col] = df[col].astype(str).str.strip().str.upper()

for col in ["is_holiday_month", "is_active_last_3m"]:
    df[col] = df[col].fillna(0).astype(int)
    
# Definir target e identificadores
TARGET_COLUMN = "monthly_total_revenue"
ID_COLUMNS = ["account_id"]
EXCLUDE_COLS = [TARGET_COLUMN] + ID_COLUMNS
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 23919 entries, 0 to 23918
Data columns (total 21 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   year                           23919 non-null  int64  
 1   quarter                        23919 non-null  int64  
 2   month                          23919 non-null  int64  
 3   month_index                    23919 non-null  int64  
 4   month_sin                      23919 non-null  float64
 5   month_cos                      23919 non-null  float64
 6   is_holiday_month               23919 non-null  int64  
 7   account_id                     23919 non-null  object 
 8   region                         23919 non-null  object 
 9   segment                        23919 non-null  object 
 10  revenue_lag_1                  22848 non-null  float64
 11  revenue_lag_2                  22848 non-null  float64
 12  revenue_lag_3                  22848 non-null 

In [6]:
null_report = df.isna().mean().sort_values(ascending=False)
print(null_report)

revenue_lag_12                   0.179104
revenue_lag_1                    0.044776
revenue_lag_2                    0.044776
opportunity_concentration_hhi    0.044776
avg_deal_size                    0.044776
revenue_lag_3                    0.044776
monthly_total_revenue            0.044776
rolling_3m_revenue               0.009490
avg_deal_size_rolling_3m         0.009490
account_id                       0.000000
region                           0.000000
segment                          0.000000
quarter                          0.000000
is_holiday_month                 0.000000
month_cos                        0.000000
month_sin                        0.000000
month_index                      0.000000
top_product_id                   0.000000
month                            0.000000
is_active_last_3m                0.000000
year                             0.000000
dtype: float64


In [7]:
# 3. Separar features y target
# Eliminado filas con valores nulos en el TARGET
df = df[~df[TARGET_COLUMN].isna()]

# Separar features y target
features = [col for col in df.columns if col not in EXCLUDE_COLS]
X = df[features].copy()
y = df[TARGET_COLUMN].copy()

# Split temporal (NO aleatorio)
train_mask = df["year"] <= 2023
val_mask   = df["year"] == 2024
test_mask  = df["year"] == 2025

X_train, y_train = X[train_mask].copy(), y[train_mask].copy()
X_val,   y_val   = X[val_mask].copy(),   y[val_mask].copy()
X_test,  y_test  = X[test_mask].copy(),  y[test_mask].copy()

# Identificar tipos
categorical_features = X_train.select_dtypes(include=["object"]).columns.tolist()
numeric_features = X_train.select_dtypes(include=["number"]).columns.tolist()

# Encoding categórico
label_encoders = {}

for col in categorical_features:
    le = LabelEncoder()
    
    X_train[col] = le.fit_transform(X_train[col].astype(str))
    X_val[col]   = le.transform(X_val[col].astype(str))
    X_test[col]  = le.transform(X_test[col].astype(str))
    
    label_encoders[col] = le

# Escalado numérico
scaler = StandardScaler()

X_train[numeric_features] = scaler.fit_transform(X_train[numeric_features])
X_val[numeric_features]   = scaler.transform(X_val[numeric_features])
X_test[numeric_features]  = scaler.transform(X_test[numeric_features])

In [15]:
# Guardar orden de features 
feature_order = X_train.columns.tolist()

# Guardar artefactos del pipeline
import joblib
import os

os.makedirs("model", exist_ok=True)

joblib.dump(label_encoders, "model/label_encoders.pkl")
joblib.dump(scaler, "model/scaler.pkl")

joblib.dump({
    "features": feature_order,
    "categorical_features": categorical_features,
    "numeric_features": numeric_features,
    "target": TARGET_COLUMN,
    "id_columns": ID_COLUMNS
}, "model/feature_list.pkl")

['model/feature_list.pkl']

In [9]:
from catboost import CatBoostRegressor

catboost_model = CatBoostRegressor(
    iterations=1000,
    learning_rate=0.05,
    depth=6,
    l2_leaf_reg=3,
    loss_function="RMSE",
    eval_metric="RMSE",
    random_seed=42,
    verbose=100,
    early_stopping_rounds=50
)

# Entrenar usando VALIDATION (no TEST)
catboost_model.fit(
    X_train, y_train,
    eval_set=(X_val, y_val),
    use_best_model=True,
    plot=False
)

0:	learn: 122901.1680901	test: 122747.4171005	best: 122747.4171005 (0)	total: 52.2ms	remaining: 52.1s
100:	learn: 20454.7214844	test: 20981.4556411	best: 20981.4556411 (100)	total: 332ms	remaining: 2.95s
200:	learn: 19769.8607990	test: 20710.3350336	best: 20708.0693684 (192)	total: 606ms	remaining: 2.41s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 20679.20256
bestIteration = 236

Shrink model to first 237 iterations.


In [10]:
# 5. Evaluación del modelo
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

# Predecir en train y test
y_train_pred = catboost_model.predict(X_train)
y_test_pred = catboost_model.predict(X_test)

# Calcular métricas
train_mse = mean_squared_error(y_train, y_train_pred)
test_mse = mean_squared_error(y_test, y_test_pred)

train_rmse = np.sqrt(train_mse)
test_rmse = np.sqrt(test_mse)

train_mae = mean_absolute_error(y_train, y_train_pred)
test_mae = mean_absolute_error(y_test, y_test_pred)

train_r2 = r2_score(y_train, y_train_pred)
test_r2 = r2_score(y_test, y_test_pred)

# Mostrar métricas
print(f"\nMétricas de evaluación:")
print(f"{'Métrica':<20} {'Train':<15} {'Test':<15}")
print("-" * 50)
print(f"{'RMSE':<20} {train_rmse:<15.4f} {test_rmse:<15.4f}")
print(f"{'MAE':<20} {train_mae:<15.4f} {test_mae:<15.4f}")
print(f"{'R² Score':<20} {train_r2:<15.4f} {test_r2:<15.4f}")


Métricas de evaluación:
Métrica              Train           Test           
--------------------------------------------------
RMSE                 19603.4915      21305.6181     
MAE                  14299.3718      14989.0932     
R² Score             0.9767          0.9720         


In [11]:
# 5. Evaluación del modelo
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

y_train_pred = catboost_model.predict(X_train)
y_val_pred   = catboost_model.predict(X_val)
y_test_pred  = catboost_model.predict(X_test)


# Métricas TRAIN

train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
train_mae  = mean_absolute_error(y_train, y_train_pred)
train_r2   = r2_score(y_train, y_train_pred)

# Métricas VALIDATION
val_rmse = np.sqrt(mean_squared_error(y_val, y_val_pred))
val_mae  = mean_absolute_error(y_val, y_val_pred)
val_r2   = r2_score(y_val, y_val_pred)

# Métricas TEST
test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
test_mae  = mean_absolute_error(y_test, y_test_pred)
test_r2   = r2_score(y_test, y_test_pred)

# Baseline simple (media histórica)
baseline_pred = np.full_like(y_test, y_train.mean(), dtype=float)
baseline_rmse = np.sqrt(mean_squared_error(y_test, baseline_pred))

# Mostrar resultados
print("\nMétricas de evaluación del modelo")
print(f"{'Métrica':<20} {'Train':<15} {'Val':<15} {'Test':<15}")
print("-" * 65)
print(f"{'RMSE':<20} {train_rmse:<15.4f} {val_rmse:<15.4f} {test_rmse:<15.4f}")
print(f"{'MAE':<20}  {train_mae:<15.4f}  {val_mae:<15.4f}  {test_mae:<15.4f}")
print(f"{'R² Score':<20} {train_r2:<15.4f} {val_r2:<15.4f} {test_r2:<15.4f}")

print("\nBaseline (media histórica)")
print(f"RMSE baseline: {baseline_rmse:.4f}")


Métricas de evaluación del modelo
Métrica              Train           Val             Test           
-----------------------------------------------------------------
RMSE                 19603.4915      20679.2024      21305.6181     
MAE                   14299.3718       15281.8359       14989.0932     
R² Score             0.9767          0.9738          0.9720         

Baseline (media histórica)
RMSE baseline: 128196.6447


In [14]:
import os

os.makedirs("model", exist_ok=True)

model_path = "model/catboost_revenue_model_.cbm"
catboost_model.save_model(model_path)

model_path

'model/catboost_revenue_model_.cbm'

In [19]:
import sys
!{sys.executable} -m pip install azure-ai-ml azure-identity

In [20]:
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential

ml_client = MLClient(
    DefaultAzureCredential(),
    subscription_id="ac83a554-a94e-4ab7-8018-11799fd0824b",
    resource_group_name="biai",
    workspace_name="biai-ML"
)


Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Overriding of current MeterProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented


In [18]:
from azure.ai.ml.entities import Model

model_asset = Model(
    path="model",
    name="revenue-opportunities-regressor",
    description="Modelo de regresión CatBoost para predicción de revenue",
    type="custom_model"
)

registered_model = ml_client.models.create_or_update(model_asset)

registered_model

Uploading catboost_revenue_model_.cbm (< 1 MB): 100%|██████████| 282k/282k [00:00<00:00, 9.50MB/s]




Model({'job_name': None, 'intellectual_property': None, 'system_metadata': None, 'is_anonymous': False, 'auto_increment_version': False, 'auto_delete_setting': None, 'name': 'revenue-opportunities-regressor', 'description': 'Modelo de regresión CatBoost para predicción de revenue', 'tags': {}, 'properties': {}, 'print_as_yaml': False, 'id': '/subscriptions/ac83a554-a94e-4ab7-8018-11799fd0824b/resourceGroups/biai/providers/Microsoft.MachineLearningServices/workspaces/biai-ML/models/revenue-opportunities-regressor/versions/1', 'Resource__source_path': '', 'base_path': '/mnt/batch/tasks/shared/LS_root/mounts/clusters/mlvirtualmachine/code/Users/evarela.dev', 'creation_context': <azure.ai.ml.entities._system_data.SystemData object at 0x7deca21df910>, 'serialize': <msrest.serialization.Serializer object at 0x7deca21dd480>, 'version': '1', 'latest_version': None, 'path': 'azureml://subscriptions/ac83a554-a94e-4ab7-8018-11799fd0824b/resourceGroups/biai/workspaces/biai-ML/datastores/workspaceb

In [6]:
import joblib
import os

os.makedirs("model", exist_ok=True)

joblib.dump(label_encoders, "model/label_encoders.pkl")
joblib.dump(scaler, "model/scaler.pkl")
joblib.dump(features, "model/feature_list.pkl")

['model/feature_list.pkl']